In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
from sklearn.metrics import silhouette_samples
from sklearn.metrics import pairwise_distances
import hdbscan

In [ ]:
darknet = '<DARKNET_NAME>'
START= '20250331'

In [ ]:
def get_day(start,i):
    start = datetime.strptime(start, '%Y%m%d')
    day = start+timedelta(days=i)
    day = day.strftime('%Y%m%d')
    return day

In [ ]:
def run_hdbscan(dataset,min_cluster_size = 10):
    # Pre-compute the cosine distance matrix
    cosine_distance = pairwise_distances(dataset, metric='cosine', n_jobs=-1)
    # Initialize the clusterer
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,metric='precomputed')
    # Get the labels
    cluster_labels = clusterer.fit_predict(cosine_distance.astype('float64'))
    dataset.loc[dataset.index, 'cluster'] = cluster_labels
    
    sh = silhouette_samples(X=dataset.iloc[:,0:200].values, labels=dataset.cluster.values,metric='cosine')
    dataset['sh'] = sh
    return dataset

In [ ]:
def get_sh_size_cluster(hdbscan_result,dim=200):
    sh = silhouette_samples(X=hdbscan_result.iloc[:,0:dim].values, labels=hdbscan_result.cluster.values,metric='cosine')
    hdbscan_result['sh'] = sh
    hdbscan_sh = hdbscan_result.groupby('cluster').agg(
        num_ip=('sh', 'size'),
        avg_sh=('sh', 'mean')
    )#.reset_index()
    return hdbscan_sh

In [ ]:
for i in range(154):
    current_day = get_day(START,i)
    df = pd.read_csv(f'embeddings/{darknet}/only_last_day/{darknet}_{current_day}.csv', index_col=[0])
    df = run_hdbscan(df)
    df = df[['sh','cluster']]
    df.to_csv(f'results_clusters/{darknet}/{darknet}_{current_day}.csv')